# 06 - Top 5 Species Recommendation by Environmental Cluster

## Role of This Notebook
This notebook turns the output of the cluster classifier into a concrete species recommendation. The goal remains interpretive: take the best spatial reading of the cell and translate it into plausible plants, now using environmental groups learned from the botanical catalog.

## How the Problem Is Approached
1. The predicted plant cluster is taken for each cell.
2. The cluster probabilities returned by the model are also used.
3. Each species receives a score that combines cluster affinity, light compatibility, and secondary indoor-use criteria.
4. The best 5 species per row are kept together with a short justification.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#777777',
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = ROOT / 'data' / 'processed'
predictions = pd.read_csv(PROCESSED_DIR / 'geometry_predictions.csv')
plants = pd.read_csv(PROCESSED_DIR / 'plants_encoded.csv')
cluster_profiles = pd.read_csv(PROCESSED_DIR / 'plants_cluster_profiles.csv')
predictions.head()

## 1. Operational Scoring Matrix
The score no longer depends only on a manual light class. The main affinity comes from the probability assigned by the model to each plant cluster. On top of that base, two interpretable layers are added: light fit and secondary bonuses for indoor use, flexibility, and thermal context.

Since relative humidity was removed from the pipeline, the recommendation no longer tries to reward or penalize species based on that criterion. This makes the final layer more coherent with the data actually available per cell.


### KEY - How Bridge Clusters Relate to Species
The model does not predict species directly. First, it predicts a `bridge cluster`, which represents an operational environmental family of plants. Then, within that family, the top 5 ranking orders concrete species according to cluster affinity, light fit, and secondary bonuses. In simple terms: the cluster says what type of environment the cell has, and the top 5 says which species fit best within that type.


In [ ]:
probability_columns = [col for col in predictions.columns if col.startswith('pred_cluster_') and col.endswith('_probability')]

def extract_cluster_id(col):
    value = col.replace('pred_cluster_', '').replace('_probability', '')
    if value.startswith('bridge_'):
        value = value.replace('bridge_', '')
    return int(value)

probability_lookup = {extract_cluster_id(col): col for col in probability_columns}

bridge_key_cols = ['sun_min_h_day', 'sun_max_h_day', 'lux_min', 'lux_max', 'sun_center_h_day', 'lux_center', 'plant_light_index']
if not {'bridge_cluster_id', 'bridge_cluster_label'}.issubset(plants.columns):
    if {'bridge_cluster_id', 'bridge_cluster_label'}.issubset(cluster_profiles.columns):
        bridge_map = cluster_profiles[['plant_variety_cluster', 'bridge_cluster_id', 'bridge_cluster_label']].drop_duplicates()
    else:
        bridge_profiles = cluster_profiles[bridge_key_cols].drop_duplicates().reset_index(drop=True).copy()
        bridge_profiles['bridge_cluster_id'] = bridge_profiles.index.astype(int)
        bridge_profiles['bridge_cluster_label'] = 'bridge_cluster_' + bridge_profiles['bridge_cluster_id'].astype(str)
        bridge_map = cluster_profiles[['plant_variety_cluster'] + bridge_key_cols].merge(bridge_profiles, on=bridge_key_cols, how='left')[['plant_variety_cluster', 'bridge_cluster_id', 'bridge_cluster_label']].drop_duplicates()
    plants = plants.merge(bridge_map, on='plant_variety_cluster', how='left')

required_cols = ['plant_variety_cluster', 'plant_cluster_label', 'bridge_cluster_id', 'bridge_cluster_label', 'plant_light_index', 'indoor_use_score', 'light_flexibility', 'temp_context_score', 'latin', 'common']
missing_required = [col for col in required_cols if col not in plants.columns]
if missing_required:
    raise ValueError(f'Missing required plant columns: {missing_required}')

plant_cluster_ids = plants['bridge_cluster_id'].astype(int).to_numpy()
plant_light_index = plants['plant_light_index'].fillna(plants['plant_light_index'].median()).to_numpy(dtype=float)
indoor_bonus = np.minimum(plants['indoor_use_score'].fillna(0).to_numpy(dtype=float) * 5, 15)
flex_bonus = np.minimum(plants['light_flexibility'].fillna(0).to_numpy(dtype=float) * 3, 10)
temp_bonus = np.minimum(plants['temp_context_score'].fillna(0).to_numpy(dtype=float) * 4, 8)

def build_cluster_affinity(row):
    return np.array([row[probability_lookup[c]] if c in probability_lookup else 0.0 for c in plant_cluster_ids], dtype=float)

def closeness_score_vector(value, targets, softness):
    return np.exp(-(((value - targets) / softness) ** 2))

def build_reason(row, plant_row, cluster_probability, light_fit, final_score):
    reasons = [f"The model favors the operational cluster {row['plant_cluster_pred_label']} for this cell."]
    reasons.append(f'Probability of the species cluster: {cluster_probability:.2f}.')
    if light_fit >= 0.80:
        reasons.append('The plant is well aligned with the cell's light intensity.')
    if plant_row['indoor_use_score'] >= 2:
        reasons.append('Its declared use is suitable for indoor conditions.')
    reasons.append(f'Score final: {round(final_score, 2)}.')
    return ' '.join(reasons)

def get_top_5_recommendations(row):
    cluster_affinity = build_cluster_affinity(row)
    light_fit = closeness_score_vector(row['tile_light_index'], plant_light_index, softness=0.25)
    score = (cluster_affinity * 70) + (light_fit * 15) + indoor_bonus + flex_bonus + temp_bonus
    top_indices = np.argsort(score)[-5:][::-1]

    record = {
        'ROW_ID': row['ROW_ID'],
        'TILE_ID': row['TILE_ID'],
        'spatial_label': row['spatial_label'],
        'plant_cluster_target_label': row['plant_cluster_target_label'],
        'plant_cluster_pred_label': row['plant_cluster_pred_label']
    }

    for rank, idx in enumerate(top_indices, start=1):
        plant_row = plants.iloc[idx]
        cluster_probability = cluster_affinity[idx]
        reason = build_reason(row, plant_row, cluster_probability, light_fit[idx], score[idx])
        record[f'top{rank}_species'] = plant_row['latin']
        record[f'top{rank}_common'] = plant_row['common']
        record[f'top{rank}_cluster'] = plant_row['plant_cluster_label']
        record[f'top{rank}_bridge_cluster'] = plant_row['bridge_cluster_label']
        record[f'top{rank}_score'] = round(float(score[idx]), 4)
        record[f'top{rank}_reason'] = reason

    return record

## 2. Visual Preview of One Cell
Before processing all rows, it is useful to inspect a single cell. The bar chart helps show whether the ranking is really separated or whether the species are too tied.


In [ ]:
sample_record = get_top_5_recommendations(predictions.iloc[0])
sample_preview = pd.DataFrame([sample_record])
display(sample_preview)

sample_species = [sample_record[f'top{i}_species'] for i in range(1, 6)]
sample_scores = [sample_record[f'top{i}_score'] for i in range(1, 6)]

plt.figure(figsize=(10, 4.5))
plt.barh(sample_species[::-1], sample_scores[::-1], color='#4e79a7', edgecolor='#444444', linewidth=0.6)
plt.title('Top 5 recommendation scores for one example cell')
plt.xlabel('Recommendation score')
plt.ylabel('Species')
plt.tight_layout()
plt.show()

### Suggested Reading of the Example
If the difference between `top1` and `top5` is wide, the conclusion is that the cell leads to a fairly clear recommendation. If the difference is small, the more cautious reading is that several species are reasonable and that the ranking works more as fine ordering than as a strict decision.


## 3. Building the Ranking by Row
The result is kept by `ROW_ID`, just like in the original series. `TILE_ID` remains useful as a spatial reference, but the final assembly relies on the unique identity of each row.


In [ ]:
recommendation_records = [get_top_5_recommendations(row) for _, row in predictions.iterrows()]
recommendations = pd.DataFrame(recommendation_records)
recommendations.to_csv(PROCESSED_DIR / 'tile_plant_recommendations_intermediate.csv', index=False)
recommendations.head()

## 4. Aggregated Reading of the Ranking
With all rows processed, these charts make it possible to answer useful questions such as: which species appear most often as top 1, how the highest scores are distributed, and whether some clusters tend to concentrate stronger recommendations than others.


In [ ]:
top1_species_counts = recommendations['top1_species'].value_counts().head(10).sort_values()
cluster_counts = recommendations['plant_cluster_pred_label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(top1_species_counts.index, top1_species_counts.values, color='#59a14f', edgecolor='#444444', linewidth=0.6)
axes[0].set_title('Most frequent top-1 species')
axes[0].set_xlabel('Number of cells')
axes[0].set_ylabel('Species')

axes[1].bar(cluster_counts.index, cluster_counts.values, color='#f28e2b', edgecolor='#444444', linewidth=0.6)
axes[1].set_title('Predicted plant clusters across cells')
axes[1].set_xlabel('Predicted plant cluster')
axes[1].set_ylabel('Number of cells')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

score_data = [recommendations[f'top{i}_score'].to_numpy() for i in range(1, 4)]
avg_top1_by_cluster = recommendations.groupby('plant_cluster_pred_label', as_index=False)['top1_score'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(score_data, tick_labels=['top1', 'top2', 'top3'], patch_artist=True, boxprops=dict(facecolor='#76b7b2', alpha=0.75))
axes[0].set_title('Score spread of the strongest recommendations')
axes[0].set_xlabel('Ranking position')
axes[0].set_ylabel('Recommendation score')

axes[1].bar(avg_top1_by_cluster['plant_cluster_pred_label'], avg_top1_by_cluster['top1_score'], color='#b07aa1', edgecolor='#444444', linewidth=0.6)
axes[1].set_title('Mean top-1 score by predicted cluster')
axes[1].set_xlabel('Predicted plant cluster')
axes[1].set_ylabel('Mean top-1 score')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Aggregated Conclusion of the Ranking
The final ranking already shows clear behavior. `Peperomia obtusifolia` appears as top 1 in `23714` rows and `Peperomia clusiifolia` in `11304`, confirming that a large part of the dataset falls into the bridge cluster associated with that profile. In addition, the average `top1_score` is around `98.98`, with the best average in `bridge_cluster_1` (`103.13`) and the lowest in `bridge_cluster_2` (`98.43`). The correct reading is that the recommendation is stable, although some areas of the dataset produce even stronger rankings than others.


### Methodological Implications of Notebook 06
This notebook changes the recommendation layer in three key ways:

- the main affinity is no longer decided only by a manual light taxonomy
- the classifier probability over bridge clusters directly participates in the ranking
- the final recommendation remains interpretable because the bonuses and reasons are explicitly traceable

This way, the output remains coherent with the approach of the reference notebook and with the narrative of the 01-07 pipeline.
